# Validação da entidade candidata Prefixo CEP

## Introdução

As etapas anteriores identificaram dois relacionamentos N:N envolvendo Geolocalização:

- Cliente — Geolocalização;
- Vendedor — Geolocalização.

A multiplicidade decorre do uso do prefixo de CEP como atributo de associação em três estruturas distintas: Cliente, Vendedor e Geolocalização. Como um mesmo prefixo pode ocorrer em múltiplos registros de cada uma dessas estruturas, a associação direta produz multiplicidade em ambos os lados.

Esta etapa avalia a hipótese de promover **Prefixo CEP** a uma entidade conceitual própria, funcionando como elemento intermediário entre Cliente, Vendedor e Geolocalização.

A hipótese será aceita somente se a nova entidade:

- possuir identidade conceitual clara;
- puder ser construída de forma unívoca a partir dos dados;
- representar adequadamente o conceito compartilhado entre as três estruturas;
- substituir os relacionamentos N:N por relacionamentos 1:N sem perda semântica;
- evitar a criação de associações artificiais entre registros de Cliente, Vendedor e Geolocalização.

## Objetivos

- levantar os prefixos distintos presentes em Cliente, Vendedor e Geolocalização;
- analisar união, interseções e exclusividades entre as três fontes;
- construir a entidade candidata `Prefixo CEP`;
- validar sua unicidade;
- verificar a participação de cada origem na nova entidade;
- testar as cardinalidades propostas:
  - Prefixo CEP 1:N Cliente;
  - Prefixo CEP 1:N Vendedor;
  - Prefixo CEP 1:N Geolocalização;
- comparar essa solução com a alternativa de entidades associativas convencionais;
- produzir evidências para uma decisão de refinamento do modelo conceitual.

## 1. Critérios de validação

| Critério | Pergunta |
|---|---|
| Identidade | `prefixo_cep` representa um conceito próprio e distinto de Cliente, Vendedor e Geolocalização? |
| Unicidade | A entidade candidata pode possuir uma única ocorrência por prefixo? |
| Cobertura | Todos os prefixos presentes nas fontes podem ser representados pela entidade candidata? |
| Referencialidade | Cada Cliente, Vendedor e registro de Geolocalização referencia exatamente um prefixo? |
| Cardinalidade | A introdução da nova entidade converte os N:N em relações 1:N? |
| Não redundância | A nova entidade evita combinações artificiais entre registros? |
| Semântica | O conceito representa corretamente uma referência territorial agregada? |

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

## 2. Localização e carregamento dos dados

In [2]:
data_directories = (Path("data/raw"), Path("../../data/raw"))
data_dir = next((path for path in data_directories if path.is_dir()), None)

if data_dir is None:
    raise FileNotFoundError(
        "Diretório data/raw não encontrado. Consulte data/README.md para obter os dados."
    )

data_dir.resolve()

PosixPath('/home/lucas/workspace/pessoal/ecommerce-analytics-data-model/data/raw')

In [3]:
arquivos = {
    "Clientes": "olist_customers_dataset.csv",
    "Vendedores": "olist_sellers_dataset.csv",
    "Geolocalização": "olist_geolocation_dataset.csv",
}

entidades = {
    nome: pd.read_csv(data_dir / arquivo)
    for nome, arquivo in arquivos.items()
}

pd.DataFrame(
    {
        "Entidade": entidades.keys(),
        "Registros": [len(df) for df in entidades.values()],
        "Colunas": [len(df.columns) for df in entidades.values()],
    }
)

,Entidade,Registros,Colunas
0,Clientes,99441,5
1,Vendedores,3095,4
2,Geolocalização,1000163,5


## 3. Inventário de prefixos por origem

Nesta etapa são extraídos os prefixos distintos presentes em cada uma das três estruturas.

In [4]:
prefixos_clientes = pd.Index(
    entidades["Clientes"]["customer_zip_code_prefix"].dropna().unique()
)
prefixos_vendedores = pd.Index(
    entidades["Vendedores"]["seller_zip_code_prefix"].dropna().unique()
)
prefixos_geo = pd.Index(
    entidades["Geolocalização"]["geolocation_zip_code_prefix"].dropna().unique()
)

pd.DataFrame(
    {
        "Origem": ["Clientes", "Vendedores", "Geolocalização"],
        "Prefixos distintos": [
            len(prefixos_clientes),
            len(prefixos_vendedores),
            len(prefixos_geo),
        ],
    }
)

,Origem,Prefixos distintos
0,Clientes,14994
1,Vendedores,2246
2,Geolocalização,19015


## 4. União, interseções e exclusividades

A entidade candidata deverá representar a união de todos os prefixos presentes nas três origens, evitando perda de cobertura.

In [5]:
uniao_prefixos = prefixos_clientes.union(prefixos_vendedores).union(prefixos_geo)

intersecao_tres = (
    prefixos_clientes
    .intersection(prefixos_vendedores)
    .intersection(prefixos_geo)
)

somente_clientes = prefixos_clientes.difference(
    prefixos_vendedores.union(prefixos_geo)
)
somente_vendedores = prefixos_vendedores.difference(
    prefixos_clientes.union(prefixos_geo)
)
somente_geo = prefixos_geo.difference(
    prefixos_clientes.union(prefixos_vendedores)
)

clientes_e_geo = prefixos_clientes.intersection(prefixos_geo)
vendedores_e_geo = prefixos_vendedores.intersection(prefixos_geo)
clientes_e_vendedores = prefixos_clientes.intersection(prefixos_vendedores)

pd.DataFrame(
    {
        "Métrica": [
            "União total de prefixos",
            "Presentes nas três fontes",
            "Somente em Clientes",
            "Somente em Vendedores",
            "Somente em Geolocalização",
            "Compartilhados Cliente–Geolocalização",
            "Compartilhados Vendedor–Geolocalização",
            "Compartilhados Cliente–Vendedor",
        ],
        "Quantidade": [
            len(uniao_prefixos),
            len(intersecao_tres),
            len(somente_clientes),
            len(somente_vendedores),
            len(somente_geo),
            len(clientes_e_geo),
            len(vendedores_e_geo),
            len(clientes_e_vendedores),
        ],
    }
)

,Métrica,Quantidade
0,União total de prefixos,19177
1,Presentes nas três fontes,2160
2,Somente em Clientes,155
3,Somente em Vendedores,5
4,Somente em Geolocalização,4099
5,Compartilhados Cliente–Geolocalização,14837
6,Compartilhados Vendedor–Geolocalização,2239
7,Compartilhados Cliente–Vendedor,2162


## 5. Construção da entidade candidata

A entidade candidata é construída com uma única ocorrência por `prefixo_cep`, utilizando a união das três fontes.

Neste estágio, `prefixo_cep` é tratado como identificador natural candidato.

In [6]:
prefixo_cep = (
    pd.DataFrame({"prefixo_cep": uniao_prefixos})
    .sort_values("prefixo_cep")
    .reset_index(drop=True)
)

prefixo_cep.head()

,prefixo_cep
0,1001
1,1002
2,1003
3,1004
4,1005


In [7]:
pd.DataFrame(
    {
        "Registros": [len(prefixo_cep)],
        "Valores únicos": [prefixo_cep["prefixo_cep"].nunique()],
        "Duplicados": [prefixo_cep["prefixo_cep"].duplicated().sum()],
        "Ausentes": [prefixo_cep["prefixo_cep"].isna().sum()],
    }
)

,Registros,Valores únicos,Duplicados,Ausentes
0,19177,19177,0,0


## 6. Cobertura da entidade candidata

In [8]:
def cobertura(origem: pd.Index, referencia: pd.Index):
    correspondentes = origem.intersection(referencia)
    orfaos = origem.difference(referencia)
    return {
        "Valores distintos": len(origem),
        "Correspondentes": len(correspondentes),
        "Órfãos": len(orfaos),
        "Cobertura (%)": len(correspondentes) / len(origem) * 100 if len(origem) else 0,
    }

df_cobertura = pd.DataFrame(
    [
        {"Origem": "Clientes", **cobertura(prefixos_clientes, uniao_prefixos)},
        {"Origem": "Vendedores", **cobertura(prefixos_vendedores, uniao_prefixos)},
        {"Origem": "Geolocalização", **cobertura(prefixos_geo, uniao_prefixos)},
    ]
)

df_cobertura.style.format({"Cobertura (%)": "{:.2f}%"})

,Origem,Valores distintos,Correspondentes,Órfãos,Cobertura (%)
0,Clientes,14994,14994,0,100.00%
1,Vendedores,2246,2246,0,100.00%
2,Geolocalização,19015,19015,0,100.00%


## 7. Participação das entidades na nova estrutura

Cada registro de Cliente, Vendedor e Geolocalização deve possuir exatamente um prefixo de CEP preenchido para que a associação com a nova entidade seja funcional.

In [9]:
df_participacao = pd.DataFrame(
    {
        "Entidade": ["Clientes", "Vendedores", "Geolocalização"],
        "Registros": [
            len(entidades["Clientes"]),
            len(entidades["Vendedores"]),
            len(entidades["Geolocalização"]),
        ],
        "Prefixos ausentes": [
            int(entidades["Clientes"]["customer_zip_code_prefix"].isna().sum()),
            int(entidades["Vendedores"]["seller_zip_code_prefix"].isna().sum()),
            int(entidades["Geolocalização"]["geolocation_zip_code_prefix"].isna().sum()),
        ],
    }
)

df_participacao["Participação completa"] = (
    df_participacao["Prefixos ausentes"] == 0
)
df_participacao

,Entidade,Registros,Prefixos ausentes,Participação completa
0,Clientes,99441,0,True
1,Vendedores,3095,0,True
2,Geolocalização,1000163,0,True


## 8. Validação das cardinalidades propostas

A hipótese estrutural é:

- um Prefixo CEP pode estar associado a muitos Clientes;
- cada Cliente possui um único Prefixo CEP;
- um Prefixo CEP pode estar associado a muitos Vendedores;
- cada Vendedor possui um único Prefixo CEP;
- um Prefixo CEP pode possuir muitos registros de Geolocalização;
- cada registro de Geolocalização pertence a um único Prefixo CEP.

In [10]:
clientes_por_prefixo = entidades["Clientes"]["customer_zip_code_prefix"].value_counts()
vendedores_por_prefixo = entidades["Vendedores"]["seller_zip_code_prefix"].value_counts()
geo_por_prefixo = entidades["Geolocalização"]["geolocation_zip_code_prefix"].value_counts()

def resumo_multiplicidade(serie, nome):
    return {
        "Relacionamento": nome,
        "Mínimo observado": int(serie.min()),
        "Mediana": float(serie.median()),
        "Média": float(serie.mean()),
        "Máximo observado": int(serie.max()),
        "Prefixos com uma ocorrência": int((serie == 1).sum()),
        "Prefixos com múltiplas ocorrências": int((serie > 1).sum()),
    }

pd.DataFrame(
    [
        resumo_multiplicidade(clientes_por_prefixo, "Prefixo CEP → Cliente"),
        resumo_multiplicidade(vendedores_por_prefixo, "Prefixo CEP → Vendedor"),
        resumo_multiplicidade(geo_por_prefixo, "Prefixo CEP → Geolocalização"),
    ]
)

,Relacionamento,Mínimo observado,Mediana,Média,Máximo observado,Prefixos com uma ocorrência,Prefixos com múltiplas ocorrências
0,Prefixo CEP → Cliente,1,4.0,6.632053,142,3012,11982
1,Prefixo CEP → Vendedor,1,1.0,1.378005,49,1709,537
2,Prefixo CEP → Geolocalização,1,29.0,52.598633,1146,1043,17972


## 9. Prefixos sem ocorrências em determinados lados

Como a entidade candidata representa a união das três fontes, alguns prefixos podem existir sem Cliente, sem Vendedor ou sem registro de Geolocalização.

In [11]:
todos_prefixos = pd.Index(prefixo_cep["prefixo_cep"])

pd.DataFrame(
    {
        "Situação": [
            "Prefixos sem Cliente",
            "Prefixos sem Vendedor",
            "Prefixos sem Geolocalização",
        ],
        "Quantidade": [
            len(todos_prefixos.difference(prefixos_clientes)),
            len(todos_prefixos.difference(prefixos_vendedores)),
            len(todos_prefixos.difference(prefixos_geo)),
        ],
    }
)

,Situação,Quantidade
0,Prefixos sem Cliente,4183
1,Prefixos sem Vendedor,16931
2,Prefixos sem Geolocalização,162


## 10. Comparação com entidades associativas convencionais

Uma alternativa formal para resolver os N:N seria introduzir:

- `Cliente_Geolocalizacao`;
- `Vendedor_Geolocalizacao`.

Entretanto, essa estratégia criaria uma associação para cada combinação entre registros que compartilham o mesmo prefixo de CEP.

O cálculo abaixo estima quantas associações seriam geradas caso cada Cliente e cada Vendedor fossem combinados com todas as ocorrências geográficas de seu prefixo.

In [12]:
ceps_cg = clientes_por_prefixo.index.intersection(geo_por_prefixo.index)
ceps_vg = vendedores_por_prefixo.index.intersection(geo_por_prefixo.index)

clientes_geo_estimados = (
    clientes_por_prefixo.reindex(ceps_cg)
    * geo_por_prefixo.reindex(ceps_cg)
).sum()

vendedores_geo_estimados = (
    vendedores_por_prefixo.reindex(ceps_vg)
    * geo_por_prefixo.reindex(ceps_vg)
).sum()

pd.DataFrame(
    {
        "Entidade associativa hipotética": [
            "Cliente_Geolocalizacao",
            "Vendedor_Geolocalizacao",
        ],
        "Associações estimadas": [
            int(clientes_geo_estimados),
            int(vendedores_geo_estimados),
        ],
    }
)

,Entidade associativa hipotética,Associações estimadas
0,Cliente_Geolocalizacao,15083455
1,Vendedor_Geolocalizacao,435087


### Interpretação

A quantidade estimada de associações não representa vínculos geográficos individualmente conhecidos.

Ela decorre exclusivamente do produto entre o número de entidades de negócio que compartilham determinado prefixo e o número de registros geográficos existentes para o mesmo prefixo.

Portanto, uma entidade associativa convencional poderia materializar combinações artificiais e aumentar substancialmente a quantidade de registros sem acrescentar informação semântica.

## 11. Estrutura conceitual candidata

Caso a hipótese seja aprovada, os relacionamentos anteriores:

- Cliente N:N Geolocalização;
- Vendedor N:N Geolocalização;

serão substituídos por:

- Prefixo CEP 1:N Cliente;
- Prefixo CEP 1:N Vendedor;
- Prefixo CEP 1:N Geolocalização.

A nova entidade não representa endereço completo nem localização exata. Seu papel é representar o agrupamento territorial definido pelo prefixo de CEP utilizado nas três estruturas de origem.

## 12. Questões para decisão final

Após a execução do notebook, responder:

1. A união das três fontes produz uma entidade `Prefixo CEP` com identificador natural único?
2. A nova entidade oferece cobertura integral de Cliente, Vendedor e Geolocalização?
3. Cada registro das três entidades possui exatamente um prefixo preenchido?
4. Os dados sustentam relações 1:N entre Prefixo CEP e Cliente?
5. Os dados sustentam relações 1:N entre Prefixo CEP e Vendedor?
6. Os dados sustentam relações 1:N entre Prefixo CEP e Geolocalização?
7. Existem prefixos presentes apenas em algumas fontes? Isso é compatível com participação opcional?
8. A solução com Prefixo CEP elimina os dois N:N sem perda de informação?
9. A solução evita a geração de associações artificiais que ocorreriam com entidades associativas convencionais?
10. A entidade Prefixo CEP possui significado conceitual suficiente para integrar o modelo?

## 13. Resultado da etapa

A decisão sobre a promoção de `Prefixo CEP` a entidade conceitual deve ser tomada somente após a execução e interpretação das evidências produzidas neste notebook.

Se aprovada, a alteração exigirá revisão controlada das etapas anteriores do documento de Modelagem Conceitual:

- lista de entidades;
- definição de identificadores;
- relacionamentos;
- cardinalidades e opcionalidades.

Os notebooks anteriores permanecem válidos como evidência do processo de refinamento, pois foram eles que revelaram a multiplicidade geográfica que motivou esta análise.